# **TUGAS 1 PRAKTISI PMML**

# Books to Scrape - Web Scraper & Data Analysis

# Scraping program untuk website "https://books.toscrape.com/




# **Pradytha Galuh Putranti**

**Statistika dan Sains Data**

**2304220013**

In [1]:
pip install requests beautifulsoup4

In [3]:
import requests
from bs4 import BeautifulSoup
import csv
import re
from urllib.parse import urljoin

def get_rating_number(rating_class):
    """Convert rating class to number"""
    ratings = {
        'One': 1,
        'Two': 2,
        'Three': 3,
        'Four': 4,
        'Five': 5
    }
    for key in ratings:
        if key in rating_class:
            return ratings[key]
    return 0

def scrape_book_details(book_url, base_url):
    """Scrape detailed information from individual book page"""
    try:
        response = requests.get(book_url)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')

        # Get product information table
        table = soup.find('table', class_='table table-striped')
        product_info = {}

        if table:
            rows = table.find_all('tr')
            for row in rows:
                header = row.find('th').text.strip()
                value = row.find('td').text.strip()
                product_info[header] = value

        # Get description
        description = ''
        product_desc = soup.find('div', id='product_description')
        if product_desc:
            desc_p = product_desc.find_next_sibling('p')
            if desc_p:
                description = desc_p.text.strip()

        # Get number of reviews
        num_reviews = product_info.get('Number of reviews', '0')

        # Get stock status and number
        availability = product_info.get('Availability', '')
        stock_status = 'Out of stock'
        stock_number = 0

        if availability:
            if 'In stock' in availability:
                stock_status = 'In stock'
                # Extract number from "In stock (22 available)"
                match = re.search(r'\((\d+) available\)', availability)
                if match:
                    stock_number = int(match.group(1))

        return {
            'price_excl_tax': product_info.get('Price (excl. tax)', ''),
            'price_incl_tax': product_info.get('Price (incl. tax)', ''),
            'tax': product_info.get('Tax', ''),
            'stock_status': stock_status,
            'stock_number': stock_number,
            'description': description,
            'num_reviews': num_reviews
        }
    except Exception as e:
        print(f"Error scraping book details from {book_url}: {e}")
        return None

def scrape_books(base_url='https://books.toscrape.com/'):
    """Main function to scrape all books from the website"""
    all_books = []
    page = 1

    print("Starting scraping process...")

    while True:
        # Construct URL for current page
        if page == 1:
            url = base_url + 'catalogue/page-1.html'
        else:
            url = f"{base_url}catalogue/page-{page}.html"

        print(f"Scraping page {page}...")

        try:
            response = requests.get(url)
            response.raise_for_status()
        except requests.exceptions.HTTPError:
            print(f"No more pages found. Total pages scraped: {page - 1}")
            break

        soup = BeautifulSoup(response.content, 'html.parser')
        books = soup.find_all('article', class_='product_pod')

        if not books:
            break

        for book in books:
            # Get basic info from listing page
            title_tag = book.find('h3').find('a')
            title = title_tag['title']
            book_path = title_tag['href']
            book_url = urljoin(base_url + 'catalogue/', book_path)

            # Get image URL
            img_tag = book.find('img')
            img_url = urljoin(base_url, img_tag['src']) if img_tag else ''

            # Get rating
            rating_tag = book.find('p', class_='star-rating')
            rating = get_rating_number(rating_tag['class']) if rating_tag else 0

            # Get price from listing page
            price_tag = book.find('p', class_='price_color')
            price = price_tag.text.strip() if price_tag else ''

            # Get category (from breadcrumb on detail page will be more accurate)
            # For now, we'll get it from the detail page

            print(f"  - Scraping: {title}")

            # Get detailed information
            details = scrape_book_details(book_url, base_url)

            if details:
                # Get category from detail page
                detail_response = requests.get(book_url)
                detail_soup = BeautifulSoup(detail_response.content, 'html.parser')
                breadcrumb = detail_soup.find('ul', class_='breadcrumb')
                category = ''
                if breadcrumb:
                    category_link = breadcrumb.find_all('a')
                    if len(category_link) >= 3:
                        category = category_link[2].text.strip()

                book_data = {
                    'category': category,
                    'code': book_url.split('/')[-2],  # Extract code from URL
                    'cover_url': img_url,
                    'title': title,
                    'rating': rating,
                    'price_excl_tax': details['price_excl_tax'],
                    'price_incl_tax': details['price_incl_tax'],
                    'tax': details['tax'],
                    'stock_status': details['stock_status'],
                    'stock_number': details['stock_number'],
                    'description': details['description'],
                    'num_reviews': details['num_reviews']
                }

                all_books.append(book_data)

        page += 1

    return all_books

def save_to_csv(books, filename='books_data.csv'):
    """Save scraped data to CSV file"""
    if not books:
        print("No data to save!")
        return

    fieldnames = [
        'category', 'code', 'cover_url', 'title', 'rating',
        'price_excl_tax', 'price_incl_tax', 'tax',
        'stock_status', 'stock_number', 'description', 'num_reviews'
    ]

    with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(books)

    print(f"\nData saved to {filename}")
    print(f"Total books scraped: {len(books)}")

if __name__ == "__main__":
    # Scrape all books
    books = scrape_books()

    # Save to CSV
    save_to_csv(books)

    # Display sample data
    if books:
        print("\n=== Sample Data (First Book) ===")
        first_book = books[0]
        for key, value in first_book.items():
            print(f"{key}: {value}")

Starting scraping process...
Scraping page 1...
  - Scraping: A Light in the Attic
  - Scraping: Tipping the Velvet
  - Scraping: Soumission
  - Scraping: Sharp Objects
  - Scraping: Sapiens: A Brief History of Humankind
  - Scraping: The Requiem Red
  - Scraping: The Dirty Little Secrets of Getting Your Dream Job
  - Scraping: The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull
  - Scraping: The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics
  - Scraping: The Black Maria
  - Scraping: Starving Hearts (Triangular Trade Trilogy, #1)
  - Scraping: Shakespeare's Sonnets
  - Scraping: Set Me Free
  - Scraping: Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)
  - Scraping: Rip it Up and Start Again
  - Scraping: Our Band Could Be Your Life: Scenes from the American Indie Underground, 1981-1991
  - Scraping: Olio
  - Scraping: Mesaerion: The Best Science Fiction Stories 1800-1849
  - Scraping: Libertariani

In [4]:
import pandas as pd

# Baca CSV
df = pd.read_csv('books_data.csv')

# Lihat 10 data pertama
print(df.head(10))

# Info dataset
print(f"\nTotal buku: {len(df)}")
print(f"\nKolom yang ada: {df.columns.tolist()}")

             category                                               code  \
0              Poetry                          a-light-in-the-attic_1000   
1  Historical Fiction                             tipping-the-velvet_999   
2             Fiction                                     soumission_998   
3             Mystery                                  sharp-objects_997   
4             History           sapiens-a-brief-history-of-humankind_996   
5         Young Adult                                the-requiem-red_995   
6            Business  the-dirty-little-secrets-of-getting-your-dream...   
7             Default  the-coming-woman-a-novel-based-on-the-life-of-...   
8             Default  the-boys-in-the-boat-nine-americans-and-their-...   
9              Poetry                                the-black-maria_991   

                                           cover_url  \
0  https://books.toscrape.com/media/cache/2c/da/2...   
1  https://books.toscrape.com/media/cache/26/0c/2..

# **Analisis sederhana**

In [5]:
# Buku dengan rating tertinggi
print("=== Top 10 Buku Rating Tertinggi ===")
print(df.nlargest(10, 'rating')[['title', 'rating', 'price_incl_tax']])

# Buku termurah dan termahal
print(f"\nBuku termurah: {df['price_incl_tax'].min()}")
print(f"Buku termahal: {df['price_incl_tax'].max()}")

# Kategori terbanyak
print("\n=== Jumlah Buku per Kategori ===")
print(df['category'].value_counts())

=== Top 10 Buku Rating Tertinggi ===
                                                title  rating price_incl_tax
4               Sapiens: A Brief History of Humankind       5         £54.23
12                                        Set Me Free       5         £17.46
13  Scott Pilgrim's Precious Little Life (Scott Pi...       5         £52.29
14                          Rip it Up and Start Again       5         £35.02
23                         Chase Me (Paris Nights #2)       5         £25.27
24                                         Black Dust       5         £34.53
28  Worlds Elsewhere: Journeys Around Shakespeare’...       5         £40.30
30  The Four Agreements: A Practical Guide to Pers...       5         £17.66
32                                  The Elephant Tree       5         £23.82
34                                     Sophie's World       5         £15.94

Buku termurah: £10.00
Buku termahal: £59.99

=== Jumlah Buku per Kategori ===
category
Default               152
No

In [6]:
df.to_excel('books_data.xlsx', index=False)

In [7]:
!pip install tqdm

In [10]:
import pandas as pd
import requests
import os
from tqdm import tqdm
import time

print("=" * 70)
print("  BOOK COVERS DOWNLOADER")
print("=" * 70)

print("\nLoading data from CSV...")
df = pd.read_csv('books_data.csv')
print(f"Found {len(df)} books")

folder_name = 'book_covers'
if not os.path.exists(folder_name):
    os.makedirs(folder_name)
    print(f"Created folder: {folder_name}/")
else:
    print(f"Using existing folder: {folder_name}/")

print(f"\nStarting download process...")
print("=" * 70)

success_count = 0
failed_count = 0
skipped_count = 0

# Progress bar
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Downloading"):
    cover_url = row['cover_url']
    code = row['code']
    title = row['title'][:30]  # Limit title length

    # File path
    file_path = os.path.join(folder_name, f"{code}.jpg")

    # Skip if already exists
    if os.path.exists(file_path):
        skipped_count += 1
        continue

    try:
        # Download image
        response = requests.get(cover_url, timeout=10)
        response.raise_for_status()

        # Save image
        with open(file_path, 'wb') as f:
            f.write(response.content)

        success_count += 1

        # Small delay to avoid overwhelming the server
        time.sleep(0.1)

    except Exception as e:
        failed_count += 1
        print(f"\nFailed to download: {title} - {str(e)}")

print("\n" + "=" * 70)
print("DOWNLOAD SUMMARY")
print("=" * 70)
print(f"Successfully downloaded: {success_count} images")
print(f"Skipped (already exist): {skipped_count} images")
print(f"Failed: {failed_count} images")
print(f"Total images in folder: {success_count + skipped_count}")
print(f"\nImages saved in: {os.path.abspath(folder_name)}/")

print("\n" + "=" * 70)
print("VERIFYING DOWNLOADS")
print("=" * 70)

# Check file sizes
files = os.listdir(folder_name)
total_size = sum(os.path.getsize(os.path.join(folder_name, f)) for f in files)
avg_size = total_size / len(files) if files else 0

print(f"Total files: {len(files)}")
print(f"Total size: {total_size / (1024*1024):.2f} MB")
print(f"Average file size: {avg_size / 1024:.2f} KB")

# Show sample of downloaded files
print(f"\nSample of downloaded files:")
for i, file in enumerate(files[:5]):
    file_path = os.path.join(folder_name, file)
    size = os.path.getsize(file_path) / 1024
    print(f"  {i+1}. {file} ({size:.2f} KB)")

if len(files) > 5:
    print(f"  ... and {len(files) - 5} more files")

print("\n" + "=" * 70)
print("DOWNLOAD COMPLETED!")
print("=" * 70)

# Try to download in Colab
try:
    from google.colab import files
    import zipfile

    print("\nCreating ZIP file for download...")
    zip_filename = 'book_covers.zip'

    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for file in os.listdir(folder_name):
            file_path = os.path.join(folder_name, file)
            zipf.write(file_path, file)

    print(f"ZIP file created: {zip_filename}")
    print(f"Downloading ZIP file...")
    files.download(zip_filename)
    print("Download complete!")

except ImportError:
    print("\nFiles saved locally in 'book_covers/' folder")
except Exception as e:
    print(f"\nCould not create ZIP: {str(e)}")
    print("Files are saved in 'book_covers/' folder")

  BOOK COVERS DOWNLOADER

Loading data from CSV...
Found 1000 books
Using existing folder: book_covers/

Starting download process...


Downloading: 100%|██████████| 1000/1000 [00:00<00:00, 5457.14it/s]



DOWNLOAD SUMMARY
Successfully downloaded: 0 images
Skipped (already exist): 1000 images
Failed: 0 images
Total images in folder: 1000

Images saved in: /content/book_covers/

VERIFYING DOWNLOADS
Total files: 1000
Total size: 9.22 MB
Average file size: 9.44 KB

Sample of downloaded files:
  1. the-undomestic-goddess_286.jpg (10.76 KB)
  2. drive-the-surprising-truth-about-what-motivates-us_804.jpg (7.98 KB)
  3. m-train_598.jpg (7.42 KB)
  4. do-androids-dream-of-electric-sheep-blade-runner-1_149.jpg (8.80 KB)
  5. when-you-are-engulfed-in-flames_303.jpg (8.50 KB)
  ... and 995 more files

DOWNLOAD COMPLETED!

Creating ZIP file for download...
ZIP file created: book_covers.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download complete!
